# 05 - HistGradientBoosting (TUNED)

## 1. What this model actually does
Gradient boosting builds an **ensemble of small decision trees added one at a time**. Tree #1 makes a rough prediction; tree #2 is trained to fix tree #1's errors; tree #3 fixes what's still wrong, and so on. Each tree nudges the prediction a little (`learning_rate`), so hundreds of weak trees combine into one strong model. The **"Hist"** variant first buckets each feature into ~255 bins (a histogram), which makes finding the best split much faster - that is why it trains in seconds.

Key knobs we tune:
- **`learning_rate`** - how big each tree's correction is. Lower = more careful, needs more trees; higher = faster but can overshoot.
- **`max_iter`** - maximum number of trees (boosting rounds).
- **`max_leaf_nodes`** - tree size / complexity. More leaves = each tree captures finer interactions but risks overfitting.
- **`l2_regularization`** - penalizes complex leaf values to reduce overfitting.
- **`early_stopping`** - holds out a slice of train and stops adding trees once the validation score stops improving, so we don't have to guess `max_iter` exactly.

## 2. Why it fits THIS dataset well
- **Native NaN handling:** it learns, per split, which direction missing values should go - so we feed the raw 50%-missing `eog_burst_index` with **no imputation** (no imputation bias).
- Scale-invariant and robust to the strong feature correlations.
- Captures the nonlinear class boundaries that sink linear models.

## 3. Its costs / weaknesses here
- Slightly behind CatBoost/SVM at defaults; needs a little tuning to close the gap.
- Sequential boosting can overfit if `learning_rate`/`max_iter` are too aggressive - hence early stopping + L2.

## 4. What changed vs the baseline
Baseline: `learning_rate=0.05, max_iter=400`, no regularization -> CV macro-F1 **0.813**. A grid search added `early_stopping`, larger trees and L2; best was `learning_rate=0.1, max_leaf_nodes=63, l2_regularization=1.0` -> **~0.817**.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

# This notebook lives in day9/models/ ; the data is one folder up in day9/
TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# Every column except the row id and the label is a predictor (21 physiological signals).
FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
TARGET   = 'sleep_stage'
y = train[TARGET]
print('train', train.shape, '| test', test.shape, '| #features', len(FEATURES))
print('class balance:', dict(y.value_counts().sort_index()))
train.head()

train (9000, 23) | test (5000, 22) | #features 21
class balance: {0: np.int64(2001), 1: np.int64(2442), 2: np.int64(2237), 3: np.int64(2320)}


,id,eeg_delta_power,eeg_theta_power,eeg_alpha_power,eeg_sigma_power,eeg_beta_power,eeg_gamma_power,eeg_slow_osc_power,eeg_spectral_entropy,eeg_spindle_density,...,eog_movement_density,eog_amplitude,heart_rate_mean,heart_rate_variability,respiration_rate,respiration_variability,spo2_mean,body_movement_index,eog_burst_index,sleep_stage
0,0,-1.51474,1.40728,10.33510,-1.61350,3.73081,0.99850,1.85689,-3.24253,-1.27096,...,2.65567,1.98733,1.60184,-2.49794,-0.59521,1.71154,1.93342,1.57365,-1.36230,1
1,1,-0.28998,0.89706,1.62494,2.41580,-2.70265,-0.10131,-1.68955,0.01442,-2.87943,...,4.36423,0.09942,3.38567,-0.56386,2.16016,-4.32301,1.07270,-2.43903,-0.37271,2
2,2,3.35435,0.32987,-5.41547,2.38680,-2.90584,-2.93372,-3.11713,-0.04647,1.61782,...,-3.87561,-0.87681,-2.84480,5.08383,-4.60411,0.37967,-2.06913,2.67324,NaN,3
3,3,-1.44917,-0.04374,1.71560,-1.27770,-0.19007,2.21826,1.69621,0.39756,0.00534,...,1.41415,0.39275,0.55060,-2.12910,2.32790,0.78319,0.98233,1.53824,-0.25040,1
4,4,1.35898,-2.36720,-7.40779,5.31815,-2.55954,-5.13284,-5.26634,1.73985,1.04618,...,-0.55616,0.86588,-1.96343,4.30036,0.22130,-1.44020,1.35760,-3.07701,-1.04947,3


## 5. Preprocessing
**None.** Unlike SVM/kNN, HistGradientBoosting (a) accepts NaN directly and (b) is scale-invariant (trees split on thresholds, not distances). So we pass the raw feature matrix, NaNs and all. This is the cleanest possible treatment of the half-missing column - no values are invented.

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier

def build_hgb(**kw):
    params = dict(random_state=42, early_stopping=True,
                  validation_fraction=0.15, n_iter_no_change=30)
    params.update(kw)
    return HistGradientBoostingClassifier(**params)

# tuned configuration
build_hgb(learning_rate=0.1, max_iter=600, max_leaf_nodes=63, l2_regularization=1.0)

,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",600
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",63
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",True
,"validation_fraction validation_fraction: int or float or None, default=0.1Proportion (or absolute size) of training data to set aside asvalidation data for early stopping. If None, early stopping is done onthe training data.The value is ignored if either early stopping is not performed, e.g.`early_stopping=False`, or if `X_val` and `y_val` are passed to fit.",0.15
,"n_iter_no_change n_iter_no_change: int, default=10Used to determine when to ""early stop"". The fitting process isstopped when none of the last ``n_iter_no_change`` scores are betterthan the ``n_iter_no_change - 1`` -th-to-last one, up to sometolerance. Only used if early stopping is performed.",30
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20


In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# ---- The validation protocol (identical for every model so scores are comparable) ----
# Why cross-validation: the test set has no labels, so we cannot score on it locally.
# Instead we repeatedly hold out part of the TRAIN set and score there.
#   StratifiedKFold(5)  -> split the 9000 rows into 5 equal parts ("folds").
#   stratified          -> each fold keeps the same 22-27% class proportions as the whole.
#   shuffle + seed 42   -> rows are shuffled once, reproducibly, before splitting.
# cross_val_score then runs 5 times: train on 4 folds (~7200 rows), score the held-out
# fold (~1800 rows the model never saw), and average the 5 held-out scores.
#   scoring='f1_macro' -> the competition metric: F1 computed per class, then averaged
#                         with EQUAL weight, so the worst class matters as much as the best.
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def benchmark(model, X, label):
    s = cross_val_score(model, X, y, cv=cv, scoring='f1_macro')
    print(f'{label}: macro-F1 = {s.mean():.4f} +/- {s.std():.4f}')
    return s

## 6. Baseline vs tuned (same 5-fold CV)
Note we feed `train[FEATURES]` directly - the NaNs are intentional and handled internally.

In [4]:
benchmark(HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, random_state=42),
          train[FEATURES], 'HistGBM baseline')
benchmark(build_hgb(learning_rate=0.1, max_iter=600, max_leaf_nodes=63, l2_regularization=1.0),
          train[FEATURES], 'HistGBM tuned')

HistGBM baseline: macro-F1 = 0.8132 +/- 0.0051


HistGBM tuned: macro-F1 = 0.8165 +/- 0.0049


array([0.81685745, 0.80929293, 0.81782441, 0.82437301, 0.81432229])

## 7. The tuning search (how the config above was chosen)
GridSearchCV over the main knobs with early stopping on. ~1-2 minutes. The leaderboard of results is printed so you can see how flat/peaked the optimum is.

In [5]:
from sklearn.model_selection import GridSearchCV
grid = {'learning_rate':[0.03,0.05,0.1], 'max_leaf_nodes':[31,63,127],
        'l2_regularization':[0.0,1.0], 'max_iter':[600]}
gs = GridSearchCV(build_hgb(), grid, scoring='f1_macro', cv=cv, n_jobs=-1)
gs.fit(train[FEATURES], y)
print('best params:', gs.best_params_, 'best macro-F1 = %.4f' % gs.best_score_)
pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')[
    ['param_learning_rate','param_max_leaf_nodes','param_l2_regularization','mean_test_score']].head(6)

best params: {'l2_regularization': 1.0, 'learning_rate': 0.1, 'max_iter': 600, 'max_leaf_nodes': 63} best macro-F1 = 0.8165


,param_learning_rate,param_max_leaf_nodes,param_l2_regularization,mean_test_score
16,0.10,63,1.0,0.816534
1,0.03,63,0.0,0.815442
15,0.10,31,1.0,0.815433
7,0.10,63,0.0,0.815033
9,0.03,31,1.0,0.815015
6,0.10,31,0.0,0.814730


## 8. Train final model on ALL data and write the submission
We refit the tuned model on every labelled row, then predict the test set and save `outputs/histgbm_submission.csv`.

In [6]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
final_hgb = build_hgb(learning_rate=0.1, max_iter=600, max_leaf_nodes=63, l2_regularization=1.0)
final_hgb.fit(train[FEATURES], y)
pred = final_hgb.predict(test[FEATURES])
submission = pd.DataFrame({'id': test['id'], 'sleep_stage': np.asarray(pred).astype(int).ravel()})
path = os.path.join(OUT_DIR, 'histgbm_submission.csv')
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
print('predicted class distribution:', dict(submission.sleep_stage.value_counts().sort_index()))
submission.head()

wrote ../outputs/histgbm_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1130), 1: np.int64(1323), 2: np.int64(1269), 3: np.int64(1278)}


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,2
4,9004,3


## 9. How to push further later
- Tune `min_samples_leaf` and `max_features`.
- Inspect per-class errors (confusion matrix) and engineer features aimed at class 2.
- Ensemble with CatBoost (similar idea) and SVM (different idea) once individual models are maxed out.